In [1]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)


Token loaded: True


In [2]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git


Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 60 (delta 23), reused 37 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 330.65 KiB | 1.31 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [3]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [4]:
!git config --global credential.helper store

In [5]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")



GitHub authentication configured.


In [6]:
!git fetch origin
!git switch nehna

branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'


In [8]:
!git status

On branch nehna
Your branch is up to date with 'origin/nehna'.

nothing to commit, working tree clean


In [9]:
!pip install -q transformers accelerate

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"   # verify this exact name on huggingface.co before running

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [11]:
def build_prompt(question, passages):

    context = "\n\n".join(
        [f"[{i+1}] {p}" for i, p in enumerate(passages)]
    )

    prompt = f"""Answer the question using ONLY the context below. Give a short, direct answer — a few words, not a full sentence. If the answer is not contained in the context, say exactly: "I don't know."

Context:
{context}

Question: {question}

Answer:"""

    return prompt

In [12]:
def generate_answer(question, passages, max_new_tokens=30):

    prompt = build_prompt(question, passages)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(llm.device)

    with torch.no_grad():

        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [13]:
import json

with open("data/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

with open("data/questions.json", encoding="utf-8") as f:
    questions = json.load(f)

with open("results/hybrid_top10.json", encoding="utf-8") as f:
    hybrid_results = json.load(f)

In [14]:
print("Corpus:", len(corpus))
print("Questions:", len(questions))
print("Hybrid results:", len(hybrid_results))

Corpus: 1000
Questions: 100
Hybrid results: 100


In [15]:
q = questions[0]

top_ids = hybrid_results[q["question_id"]][:5]

passages = [
    corpus[i]["text"]
    for i in top_ids
]

answer = generate_answer(
    q["question"],
    passages
)

print("Question:", q["question"])
print("Gold:", q["gold_answers"])
print("Model answer:", answer)

Question: What type of surnames is their a strong presence of?
Gold: ['Border Reiver', 'Border Reiver', 'Border Reiver surnames']
Model answer: Border Reiver surnames.


In [16]:
predictions = {}

for i, q in enumerate(questions):

    top_ids = hybrid_results[
        q["question_id"]
    ][:5]

    passages = [
        corpus[pid]["text"]
        for pid in top_ids
    ]

    predictions[
        q["question_id"]
    ] = generate_answer(
        q["question"],
        passages
    )

    if i % 10 == 0:
        print(f"{i}/100 done")

0/100 done
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done


In [19]:
!git status


On branch nehna
Your branch is up to date with 'origin/nehna'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/05_generation.ipynb

nothing added to commit but untracked files present (use "git add" to track)


In [22]:
!git add .

In [23]:
!git config --global user.email "nehnamehranmk17@gmail.com"
!git config --global user.name "nehnamehranmk638-dev"

In [24]:
!git commit -m "generation added"

[nehna fab995b] generation added
 1 file changed, 3816 insertions(+)
 create mode 100644 notebooks/05_generation.ipynb


In [25]:
!git push

Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 10.64 KiB | 5.32 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/nehnamehranmk638-dev/multilingual-rag-research.git
   6233942..fab995b  nehna -> nehna


In [26]:
import re
import string

def normalize_answer(text):

    text = text.lower()

    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text
    )

    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )

    text = " ".join(
        text.split()
    )

    return text

In [27]:
print(
    normalize_answer(
        "The Eiffel Tower."
    )
)

print(
    normalize_answer(
        "eiffel tower"
    )
)

eiffel tower
eiffel tower


In [28]:
def exact_match(prediction, gold_answers):

    pred_norm = normalize_answer(
        prediction
    )

    for gold in gold_answers:

        if pred_norm == normalize_answer(gold):
            return 1

    return 0

In [29]:
def f1_score(prediction, gold):

    pred_tokens = normalize_answer(
        prediction
    ).split()

    gold_tokens = normalize_answer(
        gold
    ).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(
            pred_tokens == gold_tokens
        )

    common = {}

    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1

    gold_counts = {}

    for t in gold_tokens:
        gold_counts[t] = gold_counts.get(t, 0) + 1

    overlap = 0

    for t, count in gold_counts.items():

        overlap += min(
            count,
            common.get(t, 0)
        )

    if overlap == 0:
        return 0.0

    precision = (
        overlap / len(pred_tokens)
    )

    recall = (
        overlap / len(gold_tokens)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [30]:
def best_f1(prediction, gold_answers):

    return max(
        f1_score(prediction, gold)
        for gold in gold_answers
    )

In [31]:
print(
    exact_match(
        "Paris",
        ["Paris", "the city of Paris"]
    )
)

print(
    exact_match(
        "Rome",
        ["Paris"]
    )
)

print(
    best_f1(
        "Gustave Eiffel",
        ["Eiffel"]
    )
)

1
0
0.6666666666666666


In [32]:
em_scores = []
f1_scores = []

for q in questions:

    pred = predictions[
        q["question_id"]
    ]

    em_scores.append(
        exact_match(
            pred,
            q["gold_answers"]
        )
    )

    f1_scores.append(
        best_f1(
            pred,
            q["gold_answers"]
        )
    )

print(
    "Exact Match:",
    sum(em_scores) / len(em_scores)
)

print(
    "F1:",
    sum(f1_scores) / len(f1_scores)
)

Exact Match: 0.61
F1: 0.7123936549429908


In [33]:
def contains_match(prediction, gold_answers):
    pred_norm = normalize_answer(prediction)
    for gold in gold_answers:
        if normalize_answer(gold) in pred_norm:
            return 1
    return 0

contains_scores = [contains_match(predictions[q["question_id"]], q["gold_answers"]) for q in questions]
print("Contains-match:", sum(contains_scores) / len(contains_scores))

Contains-match: 0.76


In [34]:
gold_predictions = {}
for i, q in enumerate(questions):
    gold_passage = corpus[q["gold_passage_id"]]["text"]
    gold_predictions[q["question_id"]] = generate_answer(q["question"], [gold_passage])

    if i % 10 == 0:
        print(f"{i}/100 done")

gold_em = [exact_match(gold_predictions[q["question_id"]], q["gold_answers"]) for q in questions]
gold_f1 = [best_f1(gold_predictions[q["question_id"]], q["gold_answers"]) for q in questions]

print("Gold-passage EM:", sum(gold_em) / len(gold_em))
print("Gold-passage F1:", sum(gold_f1) / len(gold_f1))

0/100 done
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
Gold-passage EM: 0.7
Gold-passage F1: 0.8016693487832007


In [35]:
refusals = sum(
    1
    for q in questions
    if "i don't know"
    in predictions[
        q["question_id"]
    ].lower()
)

print(
    "Refusals:",
    refusals,
    "out of",
    len(questions)
)

Refusals: 0 out of 100


In [36]:
import json

with open(
    "results/predictions_hybrid.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f,
        indent=2,
        ensure_ascii=False
    )

In [37]:
with open(
    "results/predictions_gold.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gold_predictions,
        f,
        indent=2,
        ensure_ascii=False
    )

In [38]:
import csv

with open(
    "results/generation_experiments.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "run_name",
        "retrieval_method",
        "model_name",
        "em",
        "f1",
        "contains_match",
        "refusals"
    ])

    writer.writerow([
        "hybrid_generation",
        "hybrid",
        "Qwen/Qwen2.5-1.5B-Instruct",
        sum(em_scores) / len(em_scores),
        sum(f1_scores) / len(f1_scores),
        sum(contains_scores) / len(contains_scores),
        refusals
    ])